# Tabelas de resultados do README

Gera as tabelas da seção 5 do [README](../../README.md) a partir das saídas do processo:

| Tabela | Seção do README | Origem |
|---|---|---|
| Conta de chegada, valores absolutos e % da receita | 5.1 | pxq (`1-Entrega/pxq/`), aba `Componentes` |
| Comparação com o Mercado Receita por família e camada, % e R$ | 5.3 | `3-Py/2-Checks/check_mr_camadas.py` |
| Tarifa homologada × tarifa realizada | 5.4 | `3-Py/2-Checks/gera_receita_faturada_bl1_vs_bl2.py` e pxq |

Saídas em `2-Checks/tabelas_readme/`: `tabelas_readme.md` (tabelas prontas para colar no README) e
`tabelas_readme.xlsx` (uma aba por tabela, com a explicação de cada uma). O texto de leitura do README
continua manual. Valores monetários em milhões de reais (MR$).

Só o Passo 0 muda: o período de cada tabela e se os checks devem ser rodados antes.

In [ ]:
import os
import sys
import subprocess
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path

import pandas as pd
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

# ---------------------------------------------------------------- Passo 0 - parameters
BASE = Path(os.environ.get("PXQ_BASE", r"C:\python_projects\claude\p_x_q\SP\BW"))

PERIODO_5_1 = ("2026-01", "2026-08")   # conta de chegada
PERIODO_5_3 = ("2026-01", "2026-05")   # comparison with the Mercado Receita (most reliable months)
PERIODO_5_4 = ("2026-01", "2026-08")   # homologated x realized tariff

# True: run check_mr_camadas.py and gera_receita_faturada_bl1_vs_bl2.py for the periods above before
# reading them (needs the pxq of every month already generated)
RODAR_CHECKS = True

SAIDA = BASE / "2-Checks" / "tabelas_readme"
SAIDA.mkdir(parents=True, exist_ok=True)
MESES = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]


def meses(periodo):
    """['2026-01', ...] for an (start, end) period."""
    return [p.strftime("%Y-%m") for p in pd.period_range(periodo[0], periodo[1], freq="M")]


def rotulo(ref):
    """'2026-03' -> 'mar'."""
    return MESES[int(ref[5:]) - 1]


def br(v, casas=1, sinal=False):
    """Brazilian number format, half-up rounding, true minus sign; '+' on request."""
    if v is None or pd.isna(v):
        return "—"
    t = Decimal(str(float(v))).quantize(Decimal(1).scaleb(-casas), ROUND_HALF_UP)
    if t == 0:
        t = abs(t)
    s = f"{t:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".").replace("-", "−")
    return ("+" + s) if (sinal and t > 0) else s


def tabela_md(df, casas=1, sinal=False, negrito=()):
    """Markdown table (index = first column), numbers in Brazilian format."""
    cab = "| " + " | ".join([df.index.name or ""] + [str(c) for c in df.columns]) + " |"
    sep = "|" + "---|" * (len(df.columns) + 1)
    linhas = [cab, sep]
    for nome, r in df.iterrows():
        cel = [br(v, casas, sinal) if not isinstance(v, str) else v for v in r]
        if nome in negrito:
            # bold the label and the numbers, not the text columns (unit)
            nome = f"**{nome}**"
            cel = [f"**{c}**" if (c != "—" and not isinstance(v, str)) else c for c, v in zip(cel, r)]
        linhas.append("| " + " | ".join([str(nome)] + cel) + " |")
    return "\n".join(linhas)


def roda(script, *args):
    """Run one of the check scripts with the same PXQ_BASE."""
    env = dict(os.environ, PXQ_BASE=str(BASE))
    r = subprocess.run([sys.executable, str(BASE / "3-Py" / "2-Checks" / script), *args],
                       cwd=BASE / "3-Py" / "2-Checks", env=env, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"{script} falhou:\n{r.stderr[-2000:]}")
    print(f"{script} {' '.join(args)}: ok")


TABELAS = []   # (sheet name, README section, explanation, DataFrame, md text)
print("periodos: 5.1", PERIODO_5_1, "| 5.3", PERIODO_5_3, "| 5.4", PERIODO_5_4)

## 5.1 — Do valor pela tarifa ao valor faturado

Conta de chegada, por mês, sobre a receita tarifável (BW + SD): (a) receita = (b) bloco 1 + (d) Tarifa
Social + (e) APE + (f) fonte incentivada + (g) 2ª passada + (h) resíduo. A linha (g1) é a 2ª passada dos
grupos de Consumo Próprio (TUSD não faturada por decisão da Empresa) e (g2) a dos demais grupos.

In [ ]:
LINHAS_5_1 = {
    "a": "(a) Receita faturada (base tarifável)", "b": "(b) Bloco 1 (tarifa cheia)",
    "c": "(c) Diferença a explicar, (b) − (a)", "d": "(d) Tarifa Social (faixa gratuita)",
    "e": "(e) Isenção APE", "f": "(f) Fonte incentivada (1ª passada)", "g": "(g) 2ª passada, líquida",
    "g1": "(g1) dela, TUSD Consumo Próprio não faturada por decisão da Empresa",
    "g2": "(g2) dela, demais grupos", "h": "(h) Resíduo sem item tarifário", "i": "(i) 2ª passada, bruta",
}
valores = {}
for ref in meses(PERIODO_5_1):
    c = pd.read_excel(BASE / "1-Entrega" / "pxq" / f"pxq_{ref}.xlsx", sheet_name="Componentes")
    c = c[c["Tarifável?"].eq("SIM")]
    col = lambda n: c[n].fillna(0.0)
    a, b = col("RECEITA DE FORNECIMENTO").sum(), col("VALOR CALCULADO (bloco 1)").sum()
    d, e = col("Ajuste Tarifa Social (R$)").sum(), col("Ajuste regime APE (R$)").sum()
    f, g = col("Ajuste 1ª passada (R$)").sum(), col("Ajuste 2ª passada (R$)")
    cp = c["Classe principal"].astype(str).eq("Consumo Próprio")
    valores[rotulo(ref)] = {"a": a, "b": b, "c": b - a, "d": d, "e": e, "f": f, "g": g.sum(),
                            "g1": g[cp].sum(), "g2": g[~cp].sum(), "h": a - b - d - e - f - g.sum(),
                            "i": g.abs().sum()}
t = pd.DataFrame(valores) / 1e6

t51a = t.rename(index=LINHAS_5_1)
t51a.insert(0, "Unidade", "R$ milhões")
t51a.index.name = "Item"
pct = (t.drop(index=["a", "b"]) / t.loc["a"] * 100)
pct.index = [LINHAS_5_1[k].replace(" por decisão da Empresa", "") for k in pct.index]
t51b = pct.copy()
t51b.insert(0, "Unidade", "% de (a)")
t51b.index.name = "Item"

md_a = tabela_md(t51a, 1)
md_b = tabela_md(t51b.assign(**{k: t51b[k] for k in t51b.columns}), 2, sinal=False)
# sign on the net lines, as in the README
md_b = "\n".join(l if not (l.startswith("| (g)") or l.startswith("| (g2)")) else
                  "| " + " | ".join([p.strip() if (i < 2 or p.strip().startswith(("−", "0,00"))) else "+" + p.strip()
                                    for i, p in enumerate(l.strip("|").split("|"))]) + " |"
                  for l in md_b.split("\n"))
TABELAS.append(("5.1 valores", "5.1", "Conta de chegada em R$ milhões: o que a tarifa explica sozinha (bloco 1) "
                "e o que explica a diferença até a receita. É a tabela \"Valores absolutos\" da seção 5.1 do README.",
                t51a, md_a))
TABELAS.append(("5.1 percentual", "5.1", "As mesmas linhas em % da receita faturada (a). É a tabela \"Em relação à "
                "receita faturada\" da seção 5.1 do README.", t51b, md_b))
print(md_a); print(); print(md_b)

## 5.3 — Comparação com o Mercado Receita, por família e camada

Bloco 2 do PxQ contra o Mercado Receita, só a receita tarifável (sem tributos, bandeiras, UD/ER e demais
lançamentos sem tarifa). Percentuais mensais e acumulado sobre o valor do MR da própria linha; a última
coluna, sobre a receita total do MR no período.

In [ ]:
if RODAR_CHECKS:
    for ref in meses(PERIODO_5_3):
        roda("check_mr_bloco2.py", ref)
    roda("consolida_check_mr_bloco2.py")
    roda("check_mr_camadas.py", *PERIODO_5_3)
arq = BASE / "2-Checks" / f"consolidado_check_mr_camadas_{PERIODO_5_3[0]}_a_{PERIODO_5_3[1]}.xlsx"
x = pd.read_excel(arq, sheet_name=None)
pct = x["1_Delta_pct"].set_index("Unnamed: 0")
rs = x["2_Delta_R$"].set_index("Unnamed: 0")
mr = x["4_MR_R$"].set_index("Unnamed: 0")

FAMILIAS = ["BE · FIO A", "BE · FIO B", "BE · ENCARGOS", "BE · PERDAS", "BE · ENERGIA"]
ORDEM = FAMILIAS + ["BE · RI e OUTROS", "CVA (total)", "FIN (total)", "Total"]
fonte = {k: k for k in FAMILIAS + ["CVA (total)", "FIN (total)"]}
fonte["Total"] = "TOTAL"
cols = [c for c in pct.columns if c != "Acumulado"]
nomes = [rotulo(c) for c in cols] + ["Acumulado"]
total_mr = mr.loc["TOTAL", "Acumulado"]

t53a = pd.DataFrame(index=ORDEM, columns=nomes + ["Acumulado, % da receita do MR"], dtype=float)
t53b = pd.DataFrame(index=ORDEM, columns=nomes, dtype=float)
for k in ORDEM:
    if k == "BE · RI e OUTROS":
        t53b.loc[k] = 0.0
        continue
    s = fonte[k]
    t53a.loc[k, nomes] = (pct.loc[s, cols + ["Acumulado"]] * 100).values
    t53a.loc[k, "Acumulado, % da receita do MR"] = rs.loc[s, "Acumulado"] / total_mr * 100
    t53b.loc[k] = (rs.loc[s, cols + ["Acumulado"]] / 1e6).values
t53b.loc["Referência: receita do Mercado Receita"] = (mr.loc["TOTAL", cols + ["Acumulado"]] / 1e6).values
for df, u in ((t53a, "% do MR"), (t53b, "R$ milhões")):
    df.insert(0, "Unidade", u)
    df.index.name = "Diferença P × Q − MR"

md_a = tabela_md(t53a, 2, sinal=True, negrito=("Total",))
md_b = tabela_md(t53b, 1, negrito=("Total",))
TABELAS.append(("5.3 percentual", "5.3", "Diferença PxQ − Mercado Receita por família (base econômica) e camada "
                "(CVA, FIN), em %. Colunas mensais e Acumulado: sobre o valor do MR da própria linha. Última coluna: "
                "diferença acumulada sobre a receita total do MR. É a primeira tabela da seção 5.3 do README.",
                t53a, md_a))
TABELAS.append(("5.3 valores", "5.3", "A mesma diferença em R$ milhões, com a receita do MR como referência. É a "
                "tabela \"Em valores absolutos\" da seção 5.3 do README.", t53b, md_b))
print(md_a); print(); print(md_b)

## 5.4 — Tarifa homologada × tarifa realizada: onde estão os descontos

Diferença entre o layout Receita Faturada pela tarifa homologada e pelo bloco 2 (entrega), separada por
origem. Positivo = desconto. A tarifa homologada é o bloco 1 sem as duas parcelas que não são
reembolsadas pela CDE (isenção de encargos do APE e SCEE - Baixa Renda), que aparecem só como memória.
A 2ª passada é aberta em TUSD do Consumo Próprio não faturada e demais grupos (com o sinal desta tabela).

In [ ]:
if RODAR_CHECKS:
    roda("gera_receita_faturada_bl1_vs_bl2.py", *PERIODO_5_4)
arq = (BASE / "2-Checks" / "receita_faturada_Bl1_vs_Bl2" /
       f"consolidado_receita_faturada_Bl1_vs_Bl2_{PERIODO_5_4[0]}_a_{PERIODO_5_4[1]}.xlsx")

def soma_aba(aba):
    """Monthly total of one origin sheet of the consolidated file, R$ milhões."""
    d = pd.read_excel(arq, sheet_name=aba).dropna(subset=["Mês"])
    d = d[d["Mês"].astype(str).str.match(r"^\d{4}-\d{2}$")].set_index("Mês")
    d = d.drop(columns=[c for c in d.columns if c in ("CONSUMO", "Total I a Z", "Total geral")])
    return d.apply(pd.to_numeric, errors="coerce").fillna(0.0).sum(axis=1) / 1e3

ORIGENS = [("desconto_tarifa_social_dmr", "Tarifa Social – DMR"), ("desconto_fonte_incentivada", "Fonte incentivada"),
           ("ajuste_2a_passada", "2ª passada")]
# parts taken out of the homologated tariff (not reimbursed by the CDE): memo from each month's conciliacao
FORA = {"SCEE - Baixa Renda": "SCEE - Baixa Renda", "Isenção de encargos (APE)": "Isenção de encargos (APE)"}


def memoria(ref):
    """Memo lines of the month's conciliacao sheet, R$ milhões."""
    c = pd.read_excel(BASE / "2-Checks" / "receita_faturada_Bl1_vs_Bl2" / f"receita_faturada_Bl1_vs_Bl2_{ref}.xlsx",
                      sheet_name="conciliacao", header=None)
    return {rot: float(c.loc[c[0].eq(chave), 1].iloc[0]) / 1e6 for chave, rot in FORA.items()}
t54 = pd.DataFrame({n: soma_aba(a) for a, n in ORIGENS}).T
# 2nd pass split: Consumo Próprio groups from the pxq (sign of this table: positive = discount)
cp = {}
for ref in t54.columns:
    c = pd.read_excel(BASE / "1-Entrega" / "pxq" / f"pxq_{ref}.xlsx", sheet_name="Componentes")
    c = c[c["Tarifável?"].eq("SIM") & c["Classe principal"].astype(str).eq("Consumo Próprio")]
    cp[ref] = -c["Ajuste 2ª passada (R$)"].fillna(0.0).sum() / 1e6
total = t54.sum()
t54.loc["dela, TUSD Consumo Próprio não faturada"] = pd.Series(cp)
t54.loc["dela, demais grupos"] = t54.loc["2ª passada"] - t54.loc["dela, TUSD Consumo Próprio não faturada"]
t54.loc["Total"] = total
mem = pd.DataFrame({ref: memoria(ref) for ref in t54.columns})
t54.loc["Retiradas da tarifa homologada (memória):"] = None
for rot in FORA.values():
    t54.loc[rot] = mem.loc[rot]
t54.loc["Bloco 1 − tarifa realizada"] = total + mem.sum()
t54.columns = [rotulo(c) for c in t54.columns]
t54.index.name = ""

md_54 = tabela_md(t54.fillna(""), 1, negrito=("Total", "Bloco 1 − tarifa realizada"))
TABELAS.append(("5.4 descontos", "5.4", "Tarifa homologada − tarifa realizada (bloco 2), em R$ milhões, por origem "
                "do desconto reembolsado pela CDE. Positivo = desconto. A tarifa homologada é o bloco 1 sem a "
                "isenção de encargos do APE e sem o SCEE - Baixa Renda, que não são reembolsados pela CDE e aparecem "
                "como memória. A linha 'Bloco 1 − tarifa realizada' é a linha (c) da tabela 5.1, a menos do "
                "resíduo. É a tabela da seção 5.4 do README.", t54, md_54))
print(md_54)

## Gravar `.md` e `.xlsx`

In [ ]:
# markdown: one block per table, ready to paste in the README
partes = ["# Tabelas de resultados do README", "",
          "Gerado por `3-Py/readme/tabelas_readme.ipynb`. Valores em R$ milhões ou %, conforme a coluna Unidade.", ""]
for aba, secao, texto, _, txt in TABELAS:
    partes += [f"## {aba} (README, seção {secao})", "", texto, "", txt, ""]
(SAIDA / "tabelas_readme.md").write_text("\n".join(partes), encoding="utf-8")

# Excel: Leia-me + one sheet per table (explanation on top, table below)
arq_x = SAIDA / "tabelas_readme.xlsx"
with pd.ExcelWriter(arq_x, engine="openpyxl") as w:
    pd.DataFrame([(a, f"Seção {s}", t) for a, s, t, _, _ in TABELAS],
                 columns=["Aba", "README", "O que mostra"]).to_excel(w, sheet_name="Leia-me", index=False)
    for aba, secao, texto, df, _ in TABELAS:
        df.reset_index().to_excel(w, sheet_name=aba, index=False, startrow=3)
        ws = w.book[aba]
        ws["A1"] = f"README, seção {secao}"
        ws["A1"].font = Font(bold=True, size=11)
        ws["A2"] = texto
        ws["A2"].alignment = Alignment(wrap_text=True, vertical="top")
        ws.merge_cells(start_row=2, start_column=1, end_row=2, end_column=min(df.shape[1] + 1, 12))
        ws.row_dimensions[2].height = 60
        for cel in ws[4]:
            cel.font = Font(bold=True, color="FFFFFF")
            cel.fill = PatternFill("solid", fgColor="1F4E79")
            cel.alignment = Alignment(horizontal="center", wrap_text=True)
        ws.column_dimensions["A"].width = 58
        for j in range(2, ws.max_column + 1):
            ws.column_dimensions[get_column_letter(j)].width = 13
            for i in range(5, ws.max_row + 1):
                ws.cell(i, j).number_format = "#,##0.00" if "%" in str(df.iloc[0, 0]) else "#,##0.0"
    lm = w.book["Leia-me"]
    lm.column_dimensions["A"].width = 18
    lm.column_dimensions["B"].width = 12
    lm.column_dimensions["C"].width = 110
    for row in lm.iter_rows(min_row=2):
        row[2].alignment = Alignment(wrap_text=True, vertical="top")
print("gravado:", SAIDA / "tabelas_readme.md")
print("gravado:", arq_x)